# Gemini Batch Transcription Pipeline with Vertex AI Agent Sessions

This Colab orchestrates the batch transcription of audio segments using Gemini on Vertex AI, leveraging Vertex AI Agent Engine Sessions to maintain context history across sequential audio transmissions within each radio channel. This replaces manual rolling window history management with robust session-backed state.

In [ ]:
# @title Install dependencies
%pip install -q "google-cloud-aiplatform[adk]" loguru tqdm

In [ ]:
# @title Imports
import asyncio
from collections import defaultdict
import json
import os
import re
import sys
import time
from urllib.parse import urlparse

from google import adk
from google import genai
from google.adk import Runner
from google.adk.events import Event
from google.adk.runners import GetSessionConfig, RunConfig
from google.adk.sessions import VertexAiSessionService
from google.cloud import storage
from google.colab import auth
from google.colab import userdata
from google.genai import types
from IPython.display import display
import pandas as pd
from tqdm.auto import tqdm
import vertexai

In [ ]:
# @title Define constants and initial logging
MODEL_ID = "gemini-3.1-flash-lite"  # @param ["gemini-3.1-flash-lite", "gemini-3-flash-preview", "gemini-3.1-pro-preview"] {type:"string"}
AGENT_NAME = "radio_transcript_session_agent"
APP_NAME = "contextual_audio_pipeline"
USER_ID = "radio_transcription_worker"


GCP_PROJECT_ID = userdata.get("GCP_PROJECT_ID")
GCS_BUCKET = userdata.get("GCS_BUCKET")

PROJECT_NAME = "one_hour_pilot"  # @param {type:"string"}
GCS_INPUT_DIR = f"segmented_audio/{PROJECT_NAME}_audio"
EXPERIMENT_NAME = ""  # @param {type:"string"}
# @markdown Enable if modifications (ie, resampling/downmixing) were applied during segmentation:
AUDIO_PREPROCESSING = False  # @param {type:"boolean"}
# @markdown Enable if masking was applied during segmentation:
AUDIO_MASKING = True  # @param {type:"boolean"}

assert not (AUDIO_PREPROCESSING and AUDIO_MASKING), (
    "Cannot enable both AUDIO_PREPROCESSING and AUDIO_MASKING simultaneously."
)

# Handle Preprocessing suffix
if not (AUDIO_PREPROCESSING or AUDIO_MASKING):
    GCS_INPUT_DIR = f"{GCS_INPUT_DIR}_raw"
if AUDIO_MASKING:
    GCS_INPUT_DIR = f"{GCS_INPUT_DIR}_masked"

# Validation: Ensure required fields are filled
assert GCP_PROJECT_ID, "GCP_PROJECT_ID must be provided in Colab userdata."
assert GCS_BUCKET, "GCS_BUCKET must be provided in Colab userdata."
assert PROJECT_NAME, "PROJECT_NAME must be provided."
assert EXPERIMENT_NAME, "EXPERIMENT_NAME must be provided."

# Pipeline Control
# fmt: off
OVERWRITE_EXISTING = False  # @param {type:"boolean"}
# fmt: on

# Create a model-specific directory name
MODEL_ID_DIR = re.sub(r"[-\.]", "_", MODEL_ID)
GCS_OUTPUT_BASE = (
    f"transcripts/{PROJECT_NAME}_audio/{MODEL_ID_DIR}/{EXPERIMENT_NAME}"
)
GCP_LOCATION = "global"

# Set environment variables required by Vertex AI and ADK
os.environ["GOOGLE_GENAI_USE_VERTEXAI"] = "TRUE"
os.environ["GOOGLE_CLOUD_PROJECT"] = GCP_PROJECT_ID
os.environ["GOOGLE_CLOUD_LOCATION"] = GCP_LOCATION

# Fully qualified Vertex AI model resource name forces Vertex AI routing automatically in ADK
VERTEX_MODEL_URI = f"projects/{GCP_PROJECT_ID}/locations/{GCP_LOCATION}/publishers/google/models/{MODEL_ID}"

# Segmentation manifest path
MANIFEST_URI = f"gs://{GCS_BUCKET}/{GCS_INPUT_DIR}/batch_manifest.jsonl"
# The consistent final path for consolidated results
CONSISTENT_OUTPUT_URI = f"gs://{GCS_BUCKET}/{GCS_OUTPUT_BASE}/predictions.jsonl"

SYSTEM_PROMPT = """
Evaluate all audio specifically as VHF/UHF fire-related dispatch radio traffic. The audio likely contains mic clicks, RF static, radio hum, and possibly some unintelligible speech. The speakers use heavy jargon.

EXPECTED TERMINOLOGY:
copy, received, affirmative, affirm, proceed, responding, responding to, en-route, on-scene, in the area, available, returning, in service, got a caller, caller advising, in quarters, arrived, go ahead, back at, engine, tanker, brush, brush truck, tender, battalion, squad, ladder, tower, tower-ladder, medic, ambulance, k, branch, chopper, copter, AIQ, AOR, DO, IC, ICP, LAT, RP, SEAT, TAC, VFIRE, VLAT, patrol, rescue, station, personnel, air attack, air tactics, helispot, lead plane, strike team, control, being toned, box alarm, cancel the balance, chaparral, exposure protection, fire attack, fire boss, forward progress stopped, forward rate of spread stopped, heavy timber, left flank, light flashy fuels, rate of spread, right flank, structure defense, structure protection, structures threatened, terrain driven, wind driven, clear, clear and in service, code 1, code 2, code 3, code 4, code 33, medical call, fire alarm, commercial fire alarm, breathing problem, cardiac, heart problem, diabetic shock, mvc, trespass, harassment, 10-4, 10-7, 10-8, 10-9, 10-15, 10-20, 10-22, 10-23, 10-91, 10-97.

CRITICAL RULES:
1. Output the transcript exactly as said, with no newlines.
2. When transcribing numbers, write the digits grouped together (e.g., 100, 6333).
3. Format all unit identifiers as the unit type followed by digits (e.g., Engine 41, Battalion 2).
4. Do not continue the speech segment beyond what is spoken.
5. QUALITY GATE: Transcribe only what you hear with high acoustic certainty. If a portion of audio is obscured, noisy, or ambiguous, you MUST replace that specific portion with [UNINTELLIGIBLE]. Do not attempt to phonetically guess ambiguous noise.

TASK:
Transcribe the attached audio. Output strictly the transcript.
"""

SAFETY_SETTINGS = [
    {"category": "HARM_CATEGORY_HATE_SPEECH", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_SEXUALLY_EXPLICIT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_DANGEROUS_CONTENT", "threshold": "BLOCK_NONE"},
    {"category": "HARM_CATEGORY_HARASSMENT", "threshold": "BLOCK_NONE"},
]

GENERATION_CONFIG = {
    "temperature": 0.0,
    "max_output_tokens": 512,
}

CONCURRENCY_LIMIT = 20
MAX_RETRIES = 3
SOCKET_TIMEOUT = 180.0

logger.remove()
logger.add(
    sys.stderr, format="<level>{level}</level>: {message}", level="WARNING"
)

In [ ]:
# @title Authenticate with GCP
auth.authenticate_user()

!gcloud config set project {GCP_PROJECT_ID} --quiet

In [ ]:
# @title Allow access to the GCS bucket to the VertexAI Service Agent
!gcloud storage buckets add-iam-policy-binding gs://{GCS_BUCKET} \
    --member="serviceAccount:service-$(gcloud projects describe {GCP_PROJECT_ID} --format='value(projectNumber)')@gcp-sa-aiplatform.iam.gserviceaccount.com" \
    --role="roles/storage.objectViewer"

In [ ]:
# @title Initialize Vertex AI Agent Engine, Session Service, and ADK Runner
vertexai.init(project=GCP_PROJECT_ID, location=GCP_LOCATION)
vertex_client = vertexai.Client(project=GCP_PROJECT_ID, location=GCP_LOCATION)

# Create an agent engine instance
agent_engine = vertex_client.agent_engines.create()
agent_engine_id = agent_engine.api_resource.name.split("/")[-1]
logger.warning(f"Active Agent Engine ID in {GCP_LOCATION}: {agent_engine_id}")

# Define ADK Agent with fully qualified Vertex AI model URI
audio_agent = adk.Agent(
    model=VERTEX_MODEL_URI,
    name=AGENT_NAME,
    instruction=SYSTEM_PROMPT,
    generate_content_config=types.GenerateContentConfig(
        temperature=GENERATION_CONFIG["temperature"],
        max_output_tokens=GENERATION_CONFIG["max_output_tokens"],
        safety_settings=SAFETY_SETTINGS,
    ),
)

# Initialize Session Service
session_service = VertexAiSessionService(
    project=GCP_PROJECT_ID,
    location=GCP_LOCATION,
    agent_engine_id=agent_engine_id,
)

# Initialize Runner
runner = Runner(
    agent=audio_agent, app_name=APP_NAME, session_service=session_service
)

storage_client = storage.Client(project=GCP_PROJECT_ID)
CHECKPOINT_FILE = "interim_backup_predictions.jsonl"
checkpoint_write_lock = asyncio.Lock()

In [ ]:
# @title Pipeline Execution Logic
def load_gcs_checkpoint() -> dict[str, str]:
    out_bucket = CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[0]
    out_path = CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[1:]
    out_path = "/".join(out_path[:-1]) + "/" + CHECKPOINT_FILE

    blob = storage_client.bucket(out_bucket).blob(out_path)
    records = {}

    if blob.exists():
        logger.info(f"Found existing checkpoint on GCS: {out_path}. Loading...")
        blob.download_to_filename(CHECKPOINT_FILE)

        with open(CHECKPOINT_FILE, "r") as f:
            for line in f:
                if line.strip():
                    record = json.loads(line)
                    if not record.get("error"):
                        records[record["audio_filepath"]] = record["transcript"]
        logger.info(f"Loaded {len(records)} completed records from GCS.")
    else:
        logger.info("No remote checkpoint found. Starting fresh.")

    return records


async def process_single_channel(
    channel_id: str,
    segment_entries: list[dict],
    completed_records: dict[str, str],
    semaphore: asyncio.Semaphore,
    pbar: tqdm,
) -> list[dict]:
    """Processes a channel using Vertex AI Agent Engine sessions to automatically maintain context."""
    results = []

    async with semaphore:
        logger.info(
            f"Starting agent session for {channel_id} ({len(segment_entries)} segments)"
        )

        channel_user_id = f"{USER_ID}_{channel_id}"
        session_id = None
        for retry in range(MAX_RETRIES):
            try:
                new_session = await session_service.create_session(
                    user_id=channel_user_id,
                    app_name=APP_NAME,
                    display_name=channel_id,
                )
                if not new_session or not new_session.id:
                    raise ValueError(f"Invalid session created: {new_session}")
                session_id = new_session.id
                break
            except Exception as e:
                logger.warning(
                    f"Attempt {retry + 1} failed to create session for {channel_id}: {e}"
                )
                await asyncio.sleep(2**retry)

        if not session_id or not new_session:
            error_msg = f"Failed to create Vertex AI session for channel {channel_id} after {MAX_RETRIES} retries."
            logger.error(error_msg)
            for entry in segment_entries:
                results.append(
                    {
                        "example_id": channel_id,
                        "audio_filepath": entry["audio_filepath"],
                        "transcript": None,
                        "error": error_msg,
                    }
                )
                pbar.update(1)
            return results

        # Grounded in empirical ASR research (https://arxiv.org/abs/2602.09044) showing optimal recognition accuracy
        # when retaining ~21.8 minutes of historical context (yielding up to 14.2% relative error reduction).
        # Calculated against the exact corpus segment mean of 2.55 seconds (~500 audio turns), retrieving the last
        # 1,000 recent events (500 audio turns + 500 transcripts) provides maximum acoustic and linguistic grounding
        # while keeping the prompt window bounded to prevent TTFT degradation on large channels.
        run_config = RunConfig(
            get_session_config=GetSessionConfig(
                create_if_not_exists=False, num_recent_events=1000
            )
        )

        try:
            for entry in segment_entries:
                uri = entry["audio_filepath"]

                if uri in completed_records:
                    cached_transcript = completed_records[uri]
                    results.append(
                        {
                            "example_id": channel_id,
                            "audio_filepath": uri,
                            "transcript": cached_transcript,
                            "error": None,
                        }
                    )

                    # Seamless Resumption: Reconstruct backend session history for completed turns
                    try:
                        user_part = types.Part.from_uri(
                            file_uri=uri, mime_type="audio/flac"
                        )
                        user_evt = Event(
                            author="user",
                            content=types.Content(
                                role="user", parts=[user_part]
                            ),
                        )
                        await session_service.append_event(
                            session=new_session, event=user_evt
                        )

                        model_part = types.Part.from_text(
                            text=cached_transcript
                        )
                        model_evt = Event(
                            author="model",
                            content=types.Content(
                                role="model", parts=[model_part]
                            ),
                        )
                        await session_service.append_event(
                            session=new_session, event=model_evt
                        )
                    except Exception as hist_err:
                        logger.warning(
                            f"Failed to reconstruct historical event for {uri}: {hist_err}"
                        )

                    pbar.update(1)
                    continue

                audio_part = types.Part.from_uri(
                    file_uri=uri, mime_type="audio/flac"
                )
                message = types.Content(role="user", parts=[audio_part])

                transcript = None
                error_msg = "Unknown error"

                async def get_streaming_response():
                    out_text = None
                    async for event in runner.run_async(
                        user_id=channel_user_id,
                        session_id=session_id,
                        new_message=message,
                        run_config=run_config,
                    ):
                        if event.is_final_response():
                            out_text = event.content.parts[0].text.strip()
                    return out_text

                for attempt in range(MAX_RETRIES):
                    try:
                        logger.debug(
                            f"[API CALL] Sending request for {uri} in session {session_id}..."
                        )
                        # Protect against silent socket hangs with a timeout guard
                        transcript = await asyncio.wait_for(
                            get_streaming_response(), timeout=SOCKET_TIMEOUT
                        )
                        if transcript:
                            logger.success(
                                f"[API SUCCESS] Received transcript for {uri}"
                            )
                            break
                    except asyncio.TimeoutError:
                        error_msg = "TimeoutError: Vertex AI streaming HTTP socket hung without response after 180s."
                        logger.warning(
                            f"[API RETRY {attempt + 1}] {uri} - {error_msg}"
                        )
                        await asyncio.sleep(2**attempt)
                    except Exception as e:
                        error_msg = f"{type(e).__name__}: {str(e)}"
                        logger.warning(
                            f"[API RETRY {attempt + 1}] {uri} - {error_msg}"
                        )
                        await asyncio.sleep(2**attempt)

                if transcript:
                    result_dict = {
                        "example_id": channel_id,
                        "audio_filepath": uri,
                        "transcript": transcript,
                        "error": None,
                    }
                    results.append(result_dict)

                    async with checkpoint_write_lock:
                        with open(CHECKPOINT_FILE, "a") as f:
                            f.write(json.dumps(result_dict) + "\n")
                else:
                    logger.error(
                        f"[API ERROR] {uri} - Final failure: {error_msg}"
                    )
                    results.append(
                        {
                            "example_id": channel_id,
                            "audio_filepath": uri,
                            "transcript": None,
                            "error": error_msg,
                        }
                    )

                pbar.update(1)
        finally:
            logger.info(
                f"Purging session {session_id} for channel {channel_id}..."
            )
            try:
                if session_id:
                    await session_service.delete_session(
                        session_id=session_id,
                        user_id=channel_user_id,
                        app_name=APP_NAME,
                    )
            except Exception as e:
                logger.warning(f"Failed to purge session {session_id}: {e}")

    return results


async def main() -> None:
    completed_records = {}

    if OVERWRITE_EXISTING:
        logger.info("OVERWRITE_EXISTING is True. Wiping outputs.")
        out_bucket_name = CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[
            0
        ]
        out_blob_path = "/".join(
            CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[1:]
        )
        out_blob = storage_client.bucket(out_bucket_name).blob(out_blob_path)
        if out_blob.exists():
            out_blob.delete()
        if os.path.exists(CHECKPOINT_FILE):
            os.remove(CHECKPOINT_FILE)
    else:
        logger.info(
            "OVERWRITE_EXISTING is False. Resuming from GCS checkpoint..."
        )
        completed_records = load_gcs_checkpoint()

    m_bucket = MANIFEST_URI.replace("gs://", "").split("/")[0]
    m_path = "/".join(MANIFEST_URI.replace("gs://", "").split("/")[1:])
    manifest_blob = storage_client.bucket(m_bucket).blob(m_path)
    if not manifest_blob.exists():
        raise FileNotFoundError(f"Manifest not found at {MANIFEST_URI}")

    content = manifest_blob.download_as_text().strip().split("\n")
    channels = defaultdict(list)
    total_segments = 0
    for line in content:
        if line.strip():
            entry = json.loads(line)
            channels[entry["example_id"]].append(entry)
            total_segments += 1

    for ch in channels:
        channels[ch].sort(key=lambda x: x.get("offset", 0))

    active_channels = {}
    for cid, entries in channels.items():
        missing = [
            e for e in entries if e["audio_filepath"] not in completed_records
        ]
        if missing:
            active_channels[cid] = entries

    if not active_channels:
        logger.info("Everything complete.")
    else:
        semaphore = asyncio.Semaphore(CONCURRENCY_LIMIT)
        with tqdm(
            total=total_segments, desc="Processing Transcriptions"
        ) as pbar:
            tasks = [
                process_single_channel(
                    cid, entries, completed_records, semaphore, pbar
                )
                for cid, entries in active_channels.items()
            ]
            await asyncio.gather(*tasks)

    if os.path.exists(CHECKPOINT_FILE):
        with open(CHECKPOINT_FILE, "r") as f:
            final_ndjson = f.read()
        out_bucket = CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[0]
        out_path = "/".join(
            CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[1:]
        )
        storage_client.bucket(out_bucket).blob(out_path).upload_from_string(
            final_ndjson
        )
        logger.info(f"Success. Results saved to {CONSISTENT_OUTPUT_URI}")

    if completed_records:
        logger.info(
            f"Pipeline Complete. Recorded {len(completed_records)} transcripts."
        )
        df = pd.DataFrame(
            list(completed_records.items()),
            columns=["audio_filepath", "transcript"],
        )
        df["example_id"] = df["audio_filepath"].apply(
            lambda x: Path(x).parent.name
        )
        display(df[["example_id", "audio_filepath", "transcript"]].head(10))


# Execute pipeline
await main()

In [ ]:
# @title Inspect Historical Session Content (Verify User & Model Event Retention)
# Inspect the event history of the active sessions to confirm audio and transcript retention
logger.warning("Fetching active sessions from Vertex AI...")
sessions_resp = await session_service.list_sessions(app_name=APP_NAME)

if sessions_resp and sessions_resp.sessions:
    sample_session = sessions_resp.sessions[-1]  # Pick the latest session
    logger.warning(
        f"Retrieving session content for Session ID: {sample_session.id} (User: {sample_session.user_id})"
    )

    session_obj = await session_service.get_session(
        user_id=sample_session.user_id,
        app_name=APP_NAME,
        session_id=sample_session.id,
    )

    if session_obj and hasattr(session_obj, "events") and session_obj.events:
        print(
            f"--- Active Session Events Summary ({len(session_obj.events)} total events) ---"
        )
        for idx, evt in enumerate(
            session_obj.events[:10]
        ):  # Inspect first 10 turns
            role = evt.author
            ts = evt.timestamp

            if role == "model":
                text_part = (
                    evt.content.parts[0].text
                    if evt.content and evt.content.parts
                    else "None"
                )
                print(
                    f"[{idx:02d}] Role: {role:<6} | Timestamp: {ts} | Transcript: {text_part}"
                )
            elif role == "user":
                uri = "Audio Clip"
                if (
                    evt.content
                    and evt.content.parts
                    and hasattr(evt.content.parts[0], "file_data")
                ):
                    uri = evt.content.parts[0].file_data.file_uri
                print(
                    f"[{idx:02d}] Role: {role:<6} | Timestamp: {ts} | Audio File: {uri}"
                )

        if len(session_obj.events) > 10:
            print(
                f"... and {len(session_obj.events) - 10} more events alternating between User (Audio) and Model (Transcript)."
            )
    else:
        print("No events recorded in this session yet.")
else:
    print(
        "No active sessions found. (Note: Sessions are automatically purged at the end of channel processing)."
    )

In [ ]:
# @title Validate Pipeline Integrity
# Count lines in manifest
m_bucket = MANIFEST_URI.replace("gs://", "").split("/")[0]
m_path = "/".join(MANIFEST_URI.replace("gs://", "").split("/")[1:])
manifest_content = (
    storage_client.bucket(m_bucket)
    .blob(m_path)
    .download_as_text()
    .strip()
    .split("\n")
)
expected_count = len([l for l in manifest_content if l.strip()])

# Count lines in output
o_bucket = CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[0]
o_path = "/".join(CONSISTENT_OUTPUT_URI.replace("gs://", "").split("/")[1:])
output_content = (
    storage_client.bucket(o_bucket)
    .blob(o_path)
    .download_as_text()
    .strip()
    .split("\n")
)
actual_count = len([l for l in output_content if l.strip()])

print(f"--- Pipeline Validation ---")
print(f"Expected Segments (Manifest): {expected_count}")
print(f"Actual Transcripts (Output):   {actual_count}")

if expected_count == actual_count:
    print("\n✅ SUCCESS: All segments were transcribed and recorded.")
else:
    print(
        f"\n❌ WARNING: Mismatch detected! Missing {expected_count - actual_count} segments."
    )